<!--
    Copyright (C) 2026 The AiQuokkaWannabe Authors <https://github.com/ai-quokka-wannabe/iso-lab>

    This program is free software: you can redistribute it and/or modify it under the terms of
    the GNU General Public License as published by the Free Software Foundation, either version
    3 of the License, or (at your option) any later version.

    This program is distributed in the hope that it will be useful, but WITHOUT ANY WARRANTY;
    without even the implied warranty of MERCHANTABILITY or FITNESS FOR A PARTICULAR PURPOSE.
    See the GNU General Public License for more details.

    You should have received a copy of the GNU General Public License along with this program.
    If not, see <https://www.gnu.org/licenses/>.
-->

# H13's Proof of Concept, Rounds 2 to 4

Rounds 2, 3 and 4 of H13's proof of concept on synthetic worms, each the script that ran in its pull request (#88, #89
and #90), verbatim, one cell each; round 1 is [h13_modular_poc.ipynb](h13_modular_poc.ipynb), whose code cells each
round runs first. The question, method and criteria of each round are in the journal of 2026-10-17, written before it
ran. numpy only; deterministic, every random number seeded.

In [1]:
import os
# The round scripts ran from the repository's root; a notebook runs from notebooks/.
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")

## Round 2: search or statistics? (#88)

In [2]:
# H13 round 2: convergence (longer and multi-start fits) against identifiability (slices of D along the four badly
# recovered parameters, others at the truth). Reuses the round-1 notebook's code cells verbatim.
import json, sys, numpy as np
cells = [c for c in json.load(open("notebooks/h13_modular_poc.ipynb"))["cells"] if c["cell_type"] == "code"]
exec("".join(cells[0]["source"])); exec("".join(cells[1]["source"]))
src = "".join(cells[2]["source"]).split("full, hist = fit(DEV, START, {})")[0]
exec(src)                                      # TRUTH_POP, DEV, HELD, START as in round 1
BAD = ["k_rd", "h_roam", "h_dwell", "v_rev"]
def report(name, p):
    err = {k: abs(np.log(p[k] / TRUTH[k])) for k in ACTIVE}
    d, s = score(p, HELD)
    print(f"{name}: held-out D {d:.3f} (s {s:.3f}); median |ln| {np.median(list(err.values())):.3f}, max {max(err.values()):.3f}; "
          + ", ".join(f"{k} {err[k]:.3f}" for k in BAD) + f"; g {p['g']:.4f}; all four within 0.4: {all(err[k] <= 0.4 for k in BAD)}")
    return d, err
d_truth, _ = score(TRUTH, HELD); print(f"truth: held-out D {d_truth:.3f}")
long, h = fit(DEV, START, {}, gens=360); report("start 0, 360 generations", long)
for oseed in (8, 9):
    p, _ = fit(DEV, START, {}, gens=120, seed=oseed); report(f"start 0, optimiser seed {oseed}, 120 generations", p)
for i, seed in enumerate((2029, 2030), 1):
    r = np.random.default_rng(seed)
    st = {k: (0.2 if k == "g" else TRUTH[k] * float(np.exp(r.normal(0, 0.5)))) for k in NAMES}
    p, _ = fit(DEV, st, {}, gens=120); report(f"start {i}, 120 generations", p)
print("slices of held-out D through the truth (64 individuals, three seeds):")
for k in BAD:
    row = []
    for f in (0.4, 0.6, 0.8, 1.0, 1.25, 1.6, 2.5):
        p = dict(TRUTH); p[k] = TRUTH[k] * f
        row.append(score(p, HELD))
    print(f"  {k}: " + ", ".join(f"x{f}: {d:.3f}±{s:.3f}" for f, (d, s) in zip((0.4, 0.6, 0.8, 1.0, 1.25, 1.6, 2.5), row)))

truth: held-out D 0.036


start 0, 360 generations: held-out D 0.067 (s 0.019); median |ln| 0.051, max 0.933; k_rd 0.807, h_roam 0.700, h_dwell 0.341, v_rev 0.933; g 0.0277; all four within 0.4: False


start 0, optimiser seed 8, 120 generations: held-out D 0.157 (s 0.024); median |ln| 0.172, max 1.019; k_rd 0.721, h_roam 1.019, h_dwell 0.221, v_rev 0.172; g 0.0450; all four within 0.4: False


start 0, optimiser seed 9, 120 generations: held-out D 0.104 (s 0.022); median |ln| 0.142, max 0.919; k_rd 0.919, h_roam 0.519, h_dwell 0.495, v_rev 0.402; g 0.0315; all four within 0.4: False


start 1, 120 generations: held-out D 0.178 (s 0.035); median |ln| 0.190, max 2.070; k_rd 0.154, h_roam 2.070, h_dwell 0.376, v_rev 0.937; g 0.0514; all four within 0.4: False


start 2, 120 generations: held-out D 0.488 (s 0.184); median |ln| 0.554, max 1.890; k_rd 0.292, h_roam 0.554, h_dwell 0.777, v_rev 0.346; g 0.0782; all four within 0.4: False
slices of held-out D through the truth (64 individuals, three seeds):


  k_rd: x0.4: 0.316±0.022, x0.6: 0.121±0.010, x0.8: 0.068±0.010, x1.0: 0.036±0.011, x1.25: 0.048±0.019, x1.6: 0.106±0.004, x2.5: 0.313±0.024


  h_roam: x0.4: 0.038±0.010, x0.6: 0.034±0.010, x0.8: 0.034±0.010, x1.0: 0.036±0.011, x1.25: 0.046±0.011, x1.6: 0.065±0.009, x2.5: 0.152±0.007


  h_dwell: x0.4: 0.185±0.035, x0.6: 0.086±0.025, x0.8: 0.042±0.016, x1.0: 0.036±0.011, x1.25: 0.074±0.011, x1.6: 0.191±0.017, x2.5: 0.687±0.044


  v_rev: x0.4: 0.305±0.012, x0.6: 0.137±0.012, x0.8: 0.049±0.015, x1.0: 0.036±0.011, x1.25: 0.066±0.007, x1.6: 0.123±0.008, x2.5: 0.261±0.014


## Round 3: statistics that separate the trade (#89)

In [3]:
# H13 round 3: the round-1 notebook's code, with five statistics added, all from the speed alone, aimed at what round 2
# found weakly seen - reversal rates within fast and within slow windows, the mean lengths of fast and of slow bouts,
# and the reverse speed - then the full fit and recovery as in round 1's criterion 1.
import json, numpy as np
cells = [c for c in json.load(open("notebooks/h13_modular_poc.ipynb"))["cells"] if c["cell_type"] == "code"]
exec("".join(cells[0]["source"])); exec("".join(cells[1]["source"]))
base_stats = stats

def extra(speed):
    S, B = speed.shape
    win = int(10 / DT); nw = S // win
    a = np.abs(speed[: nw * win]).reshape(nw, win, B).mean(1)
    high = a > 0.1                                                       # nw x B, the fast windows
    neg = speed < 0
    onset = neg & ~np.vstack([np.zeros((1, B), bool), neg[:-1]])
    pre_w = int((TAP0 - 10) / 10)                                        # windows before the tap period
    ons_w = onset[: nw * win].reshape(nw, win, B).sum(1)[:pre_w]
    hi, lo = high[:pre_w], ~high[:pre_w]
    rate_fast = (ons_w * hi).sum(0) / np.maximum(hi.sum(0) * 10 / 60, 1e-9)
    rate_slow = (ons_w * lo).sum(0) / np.maximum(lo.sum(0) * 10 / 60, 1e-9)
    def mean_bout(mask):
        out = np.zeros(B)
        for b in range(B):
            m = mask[:, b].astype(int); d = np.diff(np.r_[0, m, 0])
            lens = np.flatnonzero(d == -1) - np.flatnonzero(d == 1)
            out[b] = lens.mean() * 10 if lens.size else 0.0
        return out
    rev_speed = np.array([np.median(-speed[neg[:, b], b]) if neg[:, b].any() else 0.0 for b in range(B)])
    return np.stack([rate_fast, rate_slow, mean_bout(high), mean_bout(~high), rev_speed], 1)

def stats(speed, bend, _onset=None):
    return np.concatenate([base_stats(speed, bend), extra(speed)], 1)

exec("".join(cells[2]["source"]).split("full, hist = fit(DEV, START, {})")[0])   # the same population, groups, start
print("statistics:", DEV.shape[1])
d_truth, s_truth = score(TRUTH, HELD); print(f"truth: held-out D {d_truth:.3f} (s {s_truth:.3f})")
for gens in (120, 360):
    full, hist = fit(DEV, START, {}, gens=gens)
    err = {k: abs(np.log(full[k] / TRUTH[k])) for k in ACTIVE}
    d, s = score(full, HELD)
    med, mx = float(np.median(list(err.values()))), max(err.values())
    print(f"{gens} generations: held-out D {d:.3f} (s {s:.3f}); median |ln| {med:.3f}, max {mx:.3f}; g {full['g']:.4f}; "
          f"criterion 1 {'passes' if med <= 0.15 and mx <= 0.4 and full['g'] <= 0.02 else 'fails'}")
    print("   fit / truth:", {k: round(full[k] / TRUTH[k], 2) for k in ACTIVE})

statistics: 17


truth: held-out D 0.077 (s 0.006)


120 generations: held-out D 0.154 (s 0.034); median |ln| 0.038, max 0.348; g 0.0587; criterion 1 fails
   fit / truth: {'k_rd': 1.36, 'k_dr': 1.14, 'h_roam': 0.71, 'h_dwell': 0.76, 'd_rev': 1.02, 'v_roam': 0.96, 'v_dwell': 0.99, 'v_rev': 1.02, 'f0': 0.99, 'f1': 1.02, 'p0': 1.04, 'dep': 0.9, 'tau': 0.86}


360 generations: held-out D 0.067 (s 0.019); median |ln| 0.051, max 0.308; g 0.0312; criterion 1 fails
   fit / truth: {'k_rd': 1.36, 'k_dr': 1.15, 'h_roam': 0.9, 'h_dwell': 0.85, 'd_rev': 1.05, 'v_roam': 0.99, 'v_dwell': 0.97, 'v_rev': 1.0, 'f0': 0.98, 'f1': 1.0, 'p0': 1.03, 'dep': 1.07, 'tau': 0.82}


## Round 4: the whole loop (#90)

In [4]:
# H13 round 4: the whole loop on round 3's 17 statistics, for optimiser seeds 7, 8 and 9 - fit, ablate each module on the held-out group, drop every
# module whose ablation raises the held-out D by less than 2s, refit with the dropped modules pinned at their baseline,
# and judge criterion 1 on the refit. (Round 3's header follows.)
# H13 round 3: the round-1 notebook's code, with five statistics added, all from the speed alone, aimed at what round 2
# found weakly seen - reversal rates within fast and within slow windows, the mean lengths of fast and of slow bouts,
# and the reverse speed - then the full fit and recovery as in round 1's criterion 1.
import json, numpy as np
cells = [c for c in json.load(open("notebooks/h13_modular_poc.ipynb"))["cells"] if c["cell_type"] == "code"]
exec("".join(cells[0]["source"])); exec("".join(cells[1]["source"]))
base_stats = stats

def extra(speed):
    S, B = speed.shape
    win = int(10 / DT); nw = S // win
    a = np.abs(speed[: nw * win]).reshape(nw, win, B).mean(1)
    high = a > 0.1                                                       # nw x B, the fast windows
    neg = speed < 0
    onset = neg & ~np.vstack([np.zeros((1, B), bool), neg[:-1]])
    pre_w = int((TAP0 - 10) / 10)                                        # windows before the tap period
    ons_w = onset[: nw * win].reshape(nw, win, B).sum(1)[:pre_w]
    hi, lo = high[:pre_w], ~high[:pre_w]
    rate_fast = (ons_w * hi).sum(0) / np.maximum(hi.sum(0) * 10 / 60, 1e-9)
    rate_slow = (ons_w * lo).sum(0) / np.maximum(lo.sum(0) * 10 / 60, 1e-9)
    def mean_bout(mask):
        out = np.zeros(B)
        for b in range(B):
            m = mask[:, b].astype(int); d = np.diff(np.r_[0, m, 0])
            lens = np.flatnonzero(d == -1) - np.flatnonzero(d == 1)
            out[b] = lens.mean() * 10 if lens.size else 0.0
        return out
    rev_speed = np.array([np.median(-speed[neg[:, b], b]) if neg[:, b].any() else 0.0 for b in range(B)])
    return np.stack([rate_fast, rate_slow, mean_bout(high), mean_bout(~high), rev_speed], 1)

def stats(speed, bend, _onset=None):
    return np.concatenate([base_stats(speed, bend), extra(speed)], 1)

exec("".join(cells[2]["source"]).split("full, hist = fit(DEV, START, {})")[0])   # the same population, groups, start

passes = 0
for oseed in (7, 8, 9):
    print(f"--- optimiser seed {oseed}")
    full, _ = fit(DEV, START, {}, seed=oseed)
    d_full, s_full = score(full, HELD)
    print(f"fit: held-out D {d_full:.3f} (s {s_full:.3f})")
    dropped = []
    for mod, names in MODULES.items():
        p = dict(full); p.update({k: BASELINE[k] for k in names})
        d, s = score(p, HELD)
        keep = d - d_full >= 2 * s_full
        print(f"ablate {mod}: held-out D {d:.3f} (s {s:.3f}), delta {d - d_full:+.3f} -> {'keep' if keep else 'drop'}")
        if not keep: dropped.append(mod)
    pinned = {k: BASELINE[k] for m in dropped for k in MODULES[m]}
    print("dropped:", dropped, "- real modules dropped:", [m for m in dropped if m != "X"])
    refit, _ = fit(DEV, START, pinned, seed=oseed)
    err = {k: abs(np.log(refit[k] / TRUTH[k])) for k in ACTIVE if k not in pinned}
    d, s = score(refit, HELD)
    med, mx = float(np.median(list(err.values()))), max(err.values())
    ok = med <= 0.15 and mx <= 0.4 and refit["g"] <= 0.02 and set(dropped) <= {"X"}
    passes += ok
    print(f"refit: held-out D {d:.3f} (s {s:.3f}); median |ln| {med:.3f}, max {mx:.3f}; g {refit['g']:.4f}; criterion 1 {'passes' if ok else 'fails'}")
    print("   refit / truth:", {k: round(refit[k] / TRUTH[k], 2) for k in ACTIVE})
print(f"the loop passes criterion 1 for {passes} of 3 optimiser seeds")

--- optimiser seed 7


fit: held-out D 0.154 (s 0.034)


ablate S: held-out D 43.701 (s 1.004), delta +43.547 -> keep


ablate R: held-out D 1.473 (s 0.033), delta +1.319 -> keep


ablate U: held-out D 1.830 (s 0.135), delta +1.676 -> keep


...

ablate R: held-out D 1.447 (s 0.025), delta +1.294 -> keep


ablate U: held-out D 2.349 (s 0.128), delta +2.196 -> keep


ablate H: held-out D 0.860 (s 0.033), delta +0.707 -> keep


ablate X: held-out D 0.128 (s 0.046), delta -0.025 -> drop
dropped: ['X'] - real modules dropped: []


refit: held-out D 0.062 (s 0.022); median |ln| 0.093, max 0.416; g 0.0000; criterion 1 fails
   refit / truth: {'k_rd': 1.52, 'k_dr': 1.39, 'h_roam': 1.24, 'h_dwell': 0.69, 'd_rev': 0.99, 'v_roam': 0.99, 'v_dwell': 1.02, 'v_rev': 1.0, 'f0': 0.94, 'f1': 1.07, 'p0': 1.1, 'dep': 1.25, 'tau': 1.1}
the loop passes criterion 1 for 1 of 3 optimiser seeds
